# Wireless Networking and Mobile Computing

### 1 Duplexing for Uplink and Downlink

Duplexing controls how a system separates uplink and downlink traffic. Uplink carries data toward the network and downlink carries data toward the user. Wireless systems must isolate the two directions to avoid self-interference. The two dominant methods are time separation and frequency separation.

#### 1.1 Time Division Duplex TDD

TDD uses one frequency channel and alternates between downlink and uplink in time. A frame is divided into a downlink interval and an uplink interval, separated by a short guard period to allow radios to switch direction and settle. The ratio between downlink and uplink time can be adapted to current traffic demand.

Basic time model
$$
x(t)=
\begin{cases}
u_{\text{DL}}(t) & t\in T_{\text{DL}}[6pt]
u_{\text{UL}}(t) & t\in T_{\text{UL}}
\end{cases}
$$

If the frame duration is $T$ and the downlink portion is $\alpha T$, then the uplink portion is $(1-\alpha)T$. For example, $\alpha = 0.8$ gives a downlink-heavy ratio suitable for internet usage. A guard time $T_g$ reduces useful time slightly and enforces clean switching.

<center>
  <img src="../../Admin/multiplex/duplex_time.png" 
       alt="Time Division Duplex" 
       width="25%"
   >
</center>

#### 1.2 Frequency Division Duplex FDD

FDD allocates separate frequency bands for uplink and downlink. Transmission in both directions happens at the same time. A duplexer filters and isolates the transmit and receive bands. There is no switching delay because direction separation is done in frequency, not time.

Frequency representation
$$x_{\text{DL}}(t)=u_{\text{DL}}(t)\cos(2\pi f_{\text{DL}}t)$$
$$x_{\text{UL}}(t)=u_{\text{UL}}(t)\cos(2\pi f_{\text{UL}}t)$$

Bands are separated by a guard frequency to enable RF filter roll-off. Channel reciprocity does not hold automatically because the uplink and downlink experience different frequencies, so separate channel estimation is needed.

<center>
  <img src="../../Admin/multiplex/duplex_frequency.png" 
       alt="Frequency Division Duplex" 
       width="25%"
   >
</center>

#### 1.3 Selecting TDD vs FDD

| Criterion               | TDD                      | FDD                          |
| ----------------------- | ------------------------ | ---------------------------- |
| Separation              | Time                     | Frequency                    |
| Spectrum                | Unpaired                 | Paired                       |
| Uplink-downlink balance | Adjustable               | Fixed                        |
| Channel reciprocity     | Yes in principle         | No                           |
| Hardware                | Simpler RF front-end     | Duplex filter required       |
| Latency behavior        | Frame dependent          | Continuous                   |
| Typical use             | Wi-Fi, 5G unpaired bands | Legacy cellular paired bands |

TDD is attractive in highly asymmetric and data-driven systems. FDD remains strong where continuous low-latency bidirectional links and paired licenses already exist.

#### 1.4 Relationship to Multiplexing
Duplexing decides how to separate directions, while ,ultiplexing decides how multiple users share resources. Modern systems mix these ideas. For example, 5G uses TDD plus OFDMA plus MIMO to share time, frequency, and space across many users and services simultaneously.

### 2 Channel Multiplexing

Multiplexing lets many users or data streams share one medium by separating their transmissions in time, frequency, code, or space. A good scheme increases total carried information without letting users interfere. It also keeps latency, jitter, and energy cost within acceptable limits. In practice, systems combine several schemes at once. For example, 5G uses OFDMA across time and frequency, together with spatial multiplexing and link adaptation, while also scheduling different services with different delay and reliability targets.

#### 2.1 Time Division Multiplexing TDM

TDM divides access into repeating frames. Each frame contains slots. A user is allowed to transmit in its slot, then waits for the next frame. A shared notion of time aligns everyone so that transmissions begin at slot boundaries. A short guard time absorbs clock errors and propagation delay so that one slot does not spill into the next. The useful time per slot is therefore the slot length minus the guard.

A compact model is
$$x(t) = \sum_{k=1}^{K} u_k(t),p_k(t)$$
where $u_k(t)$ is the waveform for user $k$ and $p_k(t)$ is one when user $k$ owns the slot and zero otherwise.

Efficiency improves as synchronization improves. If the slot length is $T_s$ and the guard is $T_g$, the time efficiency for a single slot is
$$\eta_{\text{time}} = \frac{T_s - T_g}{T_s}$$

Design issues include how devices learn the time base, how the system assigns or reassigns slots as demand changes, and how large the guard must be in the face of mobility and oscillator error. TDM is attractive for deterministic latency, since a device knows exactly when it can next send and when to expect a response.

<center>
  <img src="../../Admin/multiplex/multiplex_time.png" 
       alt="Time Division Multiplex" 
       width="30%"
   >
</center>

#### 2.2 Frequency Division Multiplexing FDM

FDM separates users in frequency. Each user occupies a band with bandwidth $B$ and a center frequency $f_k$. Everyone can transmit continuously as long as bands do not overlap. Receivers select the right band and reject the rest with filters or FFT channelizers.

A simple representation is
$$x(t) = \sum_{k=1}^{K} u_k(t)\cos(2\pi f_k t + \phi_k)$$
Channel spacing must exceed the signal bandwidth plus a small guard $ \Delta f $ to account for filter roll-off and oscillator error. A rule of thumb is
$$f_{k+1} - f_k \ge B + \Delta f$$

Tighter spacing increases spectral efficiency but raises the risk of adjacent channel leakage and intermodulation. Accurate oscillators, good filtering, and compliance with spectral masks are essential. The payoff is continuous transmission with stable latency and simple scheduling. This is why FDM underpins broadcast radio, satellite transponders, microwave backhaul, and the channel structure in Wi-Fi.

<center>
  <img src="../../Admin/multiplex/multiplex_frequency.png" 
       alt="Frequency Division Multiplex" 
       width="30%"
   >
</center>

#### 2.3 Code Division Multiplexing CDM

CDM lets multiple users share the same time and the same frequency band by assigning each user a unique spreading code. Every data bit is multiplied by a faster chip sequence, so the signal’s bandwidth expands and all users overlap fully in time and frequency. Even though they occupy the same spectrum simultaneously, their signals can be separated because each code acts like a signature. At the receiver, correlating with the correct code collapses the desired user back to a narrowband signal and treats the other users as low-level interference.

A compact description is
$$x(t) = \sum_{k=1}^{K}\sum_{n} b_k[n];c_k(t - nT_c)$$
Here $b_k[n]$ is the $n$-th data symbol of user $k$, $c_k(t)$ is the chip sequence for that user, and $T_c$ is the chip period. The processing gain is
$$PG = \frac{R_{\text{chip}}}{R_{\text{data}}}$$
which quantifies how much the despreading improves the desired signal relative to uncorrelated interference.

With a chip rate of $3.84$ Mcps and a voice stream of $12.2$ kbps, $PG \approx 315$. Correlation properties of the codes and tight power control are critical. If a nearby user arrives much stronger than a distant user, the near user can swamp the far user. This is the near–far problem and it is solved with closed-loop power control and careful code management. Multipath can destroy orthogonality, so rake receivers collect energy along multiple delayed paths and add it coherently.

<center>
  <img src="../../Admin/multiplex/multiplex_code.png" 
       alt="Code Division Multiplex" 
       width="30%"
   >
</center>

### 2.4 Comparing schemes and design tradeoffs

A simple table keeps the tradeoffs straight.

| Scheme               | Separation domain  | Scheduling flexibility          | Latency behavior                     | Notable overhead                       |
| -------------------- | ------------------ | ------------------------------- | ------------------------------------ | -------------------------------------- |
| TDM                  | Time               | High, per slot and frame        | Bounded by frame length              | Guard times and control beacons        |
| FDM                  | Frequency          | Moderate, per channel plan      | Continuous if channel is reserved    | Guard bands and spectral masks         |
| CDM                  | Code               | High, add or remove codes       | Continuous, depends on power control | Code management and correlation losses |

No single scheme wins in all environments. TDM yields deterministic timing. FDM yields simple parallelism. CDM yields interference tolerance and soft capacity.